# Example: Net Present Value for a Tesla Model S
In this example, we represent the purchase, ownership, and sale of a Tesla Model S as dated signed cash flows and compute their time-0 net present value.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Construct dated net cash flows:__ Represent the purchase price, recurring ownership costs, and terminal sale value using $\bar c_j$ for $j=0,\ldots,N$.
> * __Compute accumulation and discount factors:__ Evaluate $\mathcal D_{j,0}(y;n)$ from a stated nominal annual yield $y$ and compounding frequency $n$, then invert it to obtain time-0 discount factors.
> * __Calculate and interpret NPV:__ Compute $\operatorname{NPV}_0(y)$ as the dot product of the discount-factor and net-cash-flow arrays and interpret its sign relative to the stated benchmark.

Let's construct the ownership cash flows and value them at time $0$.
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, activates the course project, and loads the required external packages.

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

  Activating project at `~/Desktop/julia_work/CHEME-5660-CourseRepository-Fall-2026`


For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 Quantitative Finance Package documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

### Problem Components and Constants
Before computing NPV, we specify the valuation convention, horizon, and cash-flow assumptions. We use the package's [`DiscreteCompoundingModel`](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.DiscreteCompoundingModel) to generate forward accumulation factors.

The package method is named `discount(...)` for compatibility, although the values it returns are forward accumulation factors: for $y>0$ they satisfy $\mathcal D_{j,0}(y;n)\geq 1$, with equality at $j=0$. Their inverses are the time-0 discount factors used in the NPV calculation.


In [2]:
compounding_model = DiscreteCompoundingModel();

Next, specify the horizon $T$, compounding frequency $n$, nominal annual yield $y$, and the remaining cash-flow assumptions. The final cash-flow index is $N=nT$.

In [3]:
T = 10.0;   # ownership horizon in years
n = 2;      # compounding intervals per year (semiannual)
y = 0.0425; # nominal annual yield used for valuation (per year)
depreciation = 0.0625; # fraction of REMAINING value lost per semiannual period

# guard the modeling assumptions before we use them -
@assert isinteger(n*T) "T = $(T) yr at n = $(n) per yr does not land on the compounding grid";
@assert 0 ≤ depreciation < 1 "depreciation must satisfy 0 ≤ depreciation < 1";

N = round(Int, n*T); # final cash-flow and compounding-period index

## Task 1: Construct the Net Cash-Flow Dictionary
Specify the cash-flow events over the Tesla Model S ownership horizon, then store the signed net cash flow $\bar c_j$ at each index $j=0,\ldots,N$.

In [4]:
purchase_price = 111630;  # what do we pay now for the Model S (USD)?
sale_price = purchase_price*(1 - depreciation)^N; # declining balance: lose `depreciation` of the REMAINING value each period
insurance_costs = 1808.0; # what does a Model S cost to insure, per semiannual period (USD)?
other_costs = 50.0;       # other ownership costs, per semiannual period (USD)
other_savings = 0.0;      # other savings, per semiannual period (USD)

In [5]:
sale_price

30704.812715234984

The `cash_flow_event_dictionary` maps each cash-flow index $j=0,\ldots,N$ to its signed net cash flow $\bar c_j$.

* At $j=0$, the purchase price is an outflow.
* For $0<j<N$, recurring insurance and other costs are netted against any savings.
* At $j=N$, the sale value and final-period savings are netted against the final-period costs.

Each line is the inner product $\bar c_j=\left\langle\mathbf c_j,\boldsymbol\nu_j\right\rangle$ introduced in the lecture. For a recurring period $0<j<N$ the components are $\mathbf c_j=(\texttt{other\_savings},\ \texttt{insurance\_costs},\ \texttt{other\_costs})$ with directions $\boldsymbol\nu_j=(+1,-1,-1)$, so $m=3$. The terminal period $j=N$ carries one additional component, the sale value, with direction $+1$.

We populate the dictionary by iterating over the cash-flow indices with a Julia [`for` loop](https://docs.julialang.org/en/v1/base/base/#for).


In [6]:
cash_flow_event_dictionary = let

    # initialize -
    cash_flow_event_dictionary = Dict{Int64,Float64}();

    # populate the signed net cash flow at each index j -
    for j ∈ 0:N
        if j == 0
            cash_flow_event_dictionary[j] = -purchase_price;
        elseif j == N
            cash_flow_event_dictionary[j] = sale_price + other_savings - (insurance_costs + other_costs);
        else
            cash_flow_event_dictionary[j] = other_savings - (insurance_costs + other_costs);
        end
    end

    cash_flow_event_dictionary
end;

## Task 2: Compute the Accumulation-Factor Dictionary
Compute $\mathcal D_{j,0}(y;n)$ for $j=0,\ldots,N$ using the stated nominal annual yield and compounding frequency. We call the package's [`discount(...)` method](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.discount-Tuple%7BAbstractCompoundingModel,%20Float64,%20Int64%7D), which retains its legacy name but returns the forward factors in a dictionary.


In [7]:
accumulation_dictionary = discount(compounding_model, y, N, λ = n);

Unhide the code block below to see $\mathcal D_{j,0}(y;n)$ and its inverse for every period. The table is constructed with [`pretty_table(...)`](https://github.com/ronisbr/PrettyTables.jl) and a [`DataFrame`](https://github.com/JuliaData/DataFrames.jl).

In [8]:
let
    df = DataFrame();
    for j ∈ 0:N
        value = accumulation_dictionary[j];
        row_df = (
            period = j,
            𝒟 = value,
            𝒟inv = 1/value,
        );
        push!(df, row_df);
    end

    pretty_table(df; table_format = TextTableFormat(borders = text_table_borders__simple))
end

========= ========= ===========
  period         𝒟       𝒟inv 
   Int64   Float64    Float64 
========= ========= ===========
       0       1.0        1.0
       1   1.02125   0.979192
       2   1.04295   0.958817
       3   1.06511   0.938866
       4   1.08775   0.919331
       5   1.11086   0.900201
       6   1.13447    0.88147
       7   1.15858   0.863129
       8    1.1832   0.845169
       9   1.20834   0.827583
      10   1.23402   0.810362
      11   1.26024   0.793501
      12   1.28702    0.77699
      13   1.31437   0.760822
      14    1.3423   0.744991
      15   1.37082   0.729489
      16   1.39995    0.71431
      17    1.4297   0.699447
      18   1.46008   0.684893
      19   1.49111   0.670642
      20   1.52279   0.656687
========= ========= ===========


### Check: Do We Recover the Nominal Annual Yield $y$?
For $j\geq1$, invert
$$
\mathcal D_{j,0}(y;n)=\left(1+\frac{y}{n}\right)^j
$$
to obtain
$$
\boxed{
y=n\left(\mathcal D_{j,0}^{1/j}-1\right).
}
$$

Iterate over the accumulation-factor dictionary, recover $y$ at each positive index, and compare it with the specified value using Julia's [`@assert`](https://docs.julialang.org/en/v1/base/base/#Base.@assert) and [`isapprox(...)`](https://docs.julialang.org/en/v1/base/math/#Base.isapprox).


In [9]:
let
    for (j, 𝒟ⱼ) ∈ accumulation_dictionary
        if j == 0
            @assert isapprox(𝒟ⱼ, 1.0) "𝒟(0,0) must equal 1, got $(𝒟ⱼ)"; # a valid accumulation factor satisfies 𝒟(0,0) = 1
            continue; # y is not recoverable at j = 0, so move on
        end

        yⱼ = n*(𝒟ⱼ^(1/j) - 1); # invert 𝒟(j,0) = (1 + y/n)^j
        @assert isapprox(y, yⱼ, rtol = 1e-4) "recovered y = $(yⱼ) at j = $(j), expected $(y)";
    end
end

## Task 3: Compute Net Present Value
The time-0 NPV is
$$
\boxed{
\operatorname{NPV}_0(y)
=\sum_{j=0}^{N}\mathcal D_{j,0}^{-1}(y;n)\bar c_j
=\left\langle\mathcal D_{\star,0}^{-1}(y;n),\bar{\mathbf c}\right\rangle.
}
$$

Convert the dictionaries to aligned arrays and use [`dot(...)`](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.dot) to evaluate the scalar product.


#### Construct the Discount-Factor Array $\mathcal D^{-1}_{\star,0}(y;n)$
Julia arrays use one-based positions, while the financial indices run from $j=0$ through $N$. At array position $i$, set $j=i-1$ and store the inverse of `accumulation_dictionary[j]`.


In [10]:
𝒟inv = let
    𝒟inv = Array{Float64,1}(undef, N+1);

    for i ∈ 1:(N+1)
        j = i - 1;
        𝒟inv[i] = 1/accumulation_dictionary[j];
    end

    𝒟inv
end;

#### Construct the Net-Cash-Flow Array $\bar{\mathbf c}$
At array position $i$, set $j=i-1$ and copy $\bar c_j$ from the cash-flow dictionary into `c̄`.


In [11]:
c̄ = let
    c̄ = Array{Float64,1}(undef, N+1);

    for i ∈ 1:(N+1)
        j = i - 1;
        c̄[i] = cash_flow_event_dictionary[j];
    end

    c̄
end;

Finally, compute $\operatorname{NPV}_0(y)$ with the [`dot(...)` method](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.dot).

In [12]:
NPV = dot(𝒟inv, c̄);
println("The time-0 NPV for a Tesla Model S over $(T) years is $(NPV) USD.")

The time-0 NPV for a Tesla Model S over 10.0 years is -121484.18779445389 USD.


### Discussion Questions
* What does a negative NPV indicate about this investment?
* What factors could we change to improve the NPV of the Tesla Model S?

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance.  Only risk capital that is not required for living expenses.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.